# Suraksha - Multiclass Model Training

**Trains XGBoost 9-class classifier for advanced fraud detection**

Person 1's Fourth Task - Run after 03_feature_engineering notebook

## Model Details:
- Algorithm: XGBoost Multiclass Classification
- Classes: 9 fraud types + 1 legitimate
- Features: 35+ behavioral + pattern features
- Output: MLflow registered model "suraksha_advanced"

**Input:** workspace.silver.upi_features  
**Output:** MLflow Model Registry → suraksha_advanced

In [0]:
%pip install xgboost scikit-learn matplotlib seaborn --quiet

In [0]:
import mlflow
import mlflow.xgboost
from pyspark.sql.functions import col
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*70)
print("🚀 SURAKSHA - MULTICLASS MODEL TRAINING")
print("="*70)

# Use default workspace experiment (serverless compute limitation)
print("\n✓ Using default MLflow experiment")

In [0]:
print("\n📊 Step 1: Loading engineered features from Silver layer...")

try:
    df = spark.read.table("workspace.silver.upi_features")
    print(f"✓ Loaded {df.count():,} transactions")
except:
    print("⚠️  Unity Catalog read failed, trying DBFS...")
    df = spark.read.format("delta").load("dbfs:/suraksha/silver/upi_features")
    print(f"✓ Loaded {df.count():,} transactions from DBFS")

# Convert to Pandas for sklearn/xgboost
df_pandas = df.toPandas()
print(f"✓ Converted to Pandas: {df_pandas.shape}")

# Show fraud distribution
print("\n📊 Fraud Type Distribution:")
print(df_pandas['fraud_type'].value_counts())

In [0]:
print("\n📊 Step 2: Selecting features for training...")

# Features to exclude (non-predictive)
exclude_cols = [
    'txn_id', 'timestamp', 'sender_vpa', 'receiver_vpa',
    'sender_id', 'receiver_id', 'device_id',
    'is_fraud', 'fraud_type', 'fraud_type_id'  # Labels
]

# Get feature columns
feature_cols = [col for col in df_pandas.columns if col not in exclude_cols]

print(f"✓ Selected {len(feature_cols)} features")
print(f"   Features: {feature_cols[:10]}...")

# Prepare X and y
X = df_pandas[feature_cols]
y = df_pandas['fraud_type_id']

print(f"\n✓ X shape: {X.shape}")
print(f"✓ y shape: {y.shape}")
print(f"✓ Classes: {sorted(y.unique())}")

In [0]:
print("\n📊 Step 3: Encoding categorical features...")

from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"✓ Found {len(categorical_cols)} categorical columns")

# Encode categoricals
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print("✓ All features encoded")

In [0]:
print("\n📊 Step 4: Splitting data (80/20)...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train set: {X_train.shape[0]:,} samples")
print(f"✓ Test set: {X_test.shape[0]:,} samples")
print(f"✓ Features: {X_train.shape[1]}")

In [0]:
print("\n📊 Step 5: Training XGBoost multiclass classifier...")

# Model parameters
params = {
    'objective': 'multi:softmax',
    'num_class': 10,  # 9 fraud types + 1 legitimate
    'max_depth': 8,
    'learning_rate': 0.1,
    'n_estimators': 200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

# Train model
print("   Training in progress...")
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train)

print("✓ Model trained!")

# Predictions
y_pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\n✓ Accuracy: {accuracy:.4f}")
print(f"✓ F1 Score: {f1:.4f}")

In [0]:
print("\n📊 Step 6: Detailed evaluation...")

# Classification report
fraud_type_names = [
    "Legitimate", "Velocity Fraud", "Mule Account", 
    "SIM Swap", "Device Takeover", "Beneficiary Manipulation",
    "Amount Anomaly", "Temporal Anomaly", "Merchant Fraud", 
    "Failed-Then-Success"
]

print("\n   Classification Report:")
print(classification_report(y_test, y_pred, target_names=fraud_type_names[:len(np.unique(y_test))]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\n✓ Confusion matrix generated")

In [0]:
print("\n📊 Step 7: Saving trained model...")

import pickle
import os

# Create model directory
model_dir = "/Workspace/Users/vinodekdhoke@gmail.com/Suraksha/models/"
os.makedirs(model_dir, exist_ok=True)

# Save model
model_path = model_dir + "suraksha_advanced.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"✓ Model saved: {model_path}")
print("✓ Ready for serving!")

In [0]:
print("\n" + "="*70)
print("✅ MODEL TRAINING COMPLETE!")
print("="*70)
print("\n📊 Model Summary:")
print(f"   • Algorithm: XGBoost Multiclass")
print(f"   • Classes: 10 (9 fraud types + legitimate)")
print(f"   • Features: {X_train.shape[1]}")
print(f"   • Accuracy: {accuracy:.2%}")
print(f"   • F1 Score: {f1:.4f}")
print(f"   • Model Path: /Workspace/Users/vinodekdhoke@gmail.com/Suraksha/models/suraksha_advanced.pkl")

print("\n🎯 Next Steps:")
print("   1. Build RAG pipeline (shared/rag_pipeline)")
print("   2. Create model serving endpoint (06_model_serving)")
print("   3. Build Streamlit app for demo")

print("\n" + "="*70)

In [0]:
print("\n📊 Extracting feature names from trained model...")

import pickle

# Load the saved model
model_path = "/Workspace/Users/vinodekdhoke@gmail.com/Suraksha/models/suraksha_advanced.pkl"
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Get feature names from the model
feature_names = loaded_model.get_booster().feature_names

print(f"\n✓ Model has {len(feature_names)} features:")
print("\nMODEL_FEATURES = [")
for i, feat in enumerate(feature_names):
    print(f"    '{feat}',")
print("]")  

print(f"\n✓ Feature extraction complete!")